# Agentic Math Solver — runner Kaggle (2x T4)

Clona o repo, sobe o vLLM (GPU 0) servindo o modelo, sobe o site Flask (GPU 1 livre pro `easyocr`) e expõe tudo publicamente via Cloudflare Tunnel.

Rode as células em ordem. A última fica em primeiro plano e imprime uma URL `https://*.trycloudflare.com` — abra no navegador ou no VSCode (`Ctrl+Shift+P` → "Simple Browser: Show").

**Segurança:** essa URL fica acessível por qualquer pessoa que a tiver (não é indexada, mas também não tem senha). Não deixe rodando sem necessidade, e não compartilhe o link.

In [3]:
# Célula 1 — clona o repo e instala o pacote
%cd /kaggle/working
!rm -rf agentic-math-solver

# Se o repo for PRIVADO, essa clonagem HTTPS anônima vai falhar com "Authentication failed".
# Nesse caso: crie um GitHub Personal Access Token, salve como Kaggle Secret (ex: GITHUB_TOKEN)
# e troque a linha abaixo por:
#   from kaggle_secrets import UserSecretsClient
#   token = UserSecretsClient().get_secret("GITHUB_TOKEN")
#   !git clone https://{token}@github.com/rick0110/agentic-math-solver.git
!git clone https://github.com/rick0110/agentic-math-solver.git

%cd agentic-math-solver
!pip install -q -e .
!pip install -q bitsandbytes

# vllm é uma dependência OPCIONAL do projeto (extra "[vllm]" no pyproject.toml, pra manter
# o pacote core leve) — por isso o comando acima não traz o binário `vllm`. Instalando aqui:
# pode demorar alguns minutos e reaproveita o torch que o Kaggle já vem com CUDA configurado.
!pip install -q vllm
!which vllm || echo "AVISO: binário 'vllm' ainda não apareceu no PATH — reinicie o kernel e rode esta célula de novo."

/kaggle/working
Cloning into 'agentic-math-solver'...
remote: Enumerating objects: 113, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 113 (delta 35), reused 104 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (113/113), 99.82 KiB | 1.28 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/kaggle/working/agentic-math-solver
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for agentic-math-solver (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 1.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.7/303.7 MB 5.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 

In [ ]:
# Célula 2 — sobe o vLLM na GPU 0, servindo o modelo quantizado em 4-bit
# + speculative decoding: um modelo pequeno (1.5B) "adivinha" vários tokens à frente,
# o modelo grande (7B) verifica todos de uma vez numa única passada — sem perda de
# qualidade (distribuição de saída é idêntica à do modelo grande sozinho), só ganho
# de velocidade quando o rascunho acerta.
import os, subprocess, time, urllib.request

vllm_env = dict(os.environ)
vllm_env["CUDA_VISIBLE_DEVICES"] = "0"
# Se o build do sampler flashinfer falhar no ambiente do Kaggle (erro de JIT/CUDA toolchain),
# descomente a linha abaixo pra cair no sampler nativo do PyTorch (mais lento, porém estável):
# vllm_env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

vllm_log = open("/kaggle/working/vllm.log", "w")
vllm_proc = subprocess.Popen(
    [
        "vllm", "serve", "Qwen/Qwen2.5-Math-7B-Instruct",
        "--port", "8000",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.85",
        "--quantization", "bitsandbytes",
        "--load-format", "bitsandbytes",
        "--speculative-model", "Qwen/Qwen2.5-Math-1.5B-Instruct",
        "--num-speculative-tokens", "5",
    ],
    env=vllm_env, stdout=vllm_log, stderr=subprocess.STDOUT,
)

print("Subindo vLLM (pode levar alguns minutos na primeira vez, baixando os dois modelos)...")
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2)
        print("vLLM pronto em http://127.0.0.1:8000")
        break
    except Exception:
        time.sleep(5)
else:
    print("vLLM ainda não respondeu — confira /kaggle/working/vllm.log antes de continuar.")

# Se o log mostrar erro de incompatibilidade entre speculative decoding e a quantização
# bitsandbytes (é uma combinação ainda não tão madura no vLLM), rode de novo sem as duas
# últimas flags (--speculative-model / --num-speculative-tokens) pra isolar o problema.

In [5]:
# Célula 3 — sobe o site Flask na GPU 1 (livre pro easyocr), apontando pro vLLM local
import os, subprocess

flask_env = dict(os.environ)
flask_env["CUDA_VISIBLE_DEVICES"] = "1"
flask_env["AGEMATH_MODEL_BACKEND"] = "remote"
flask_env["AGEMATH_MODEL_ENDPOINT"] = "http://127.0.0.1:8000/v1"
flask_env["AGEMATH_MODEL_NAME"] = "Qwen/Qwen2.5-Math-7B-Instruct"
flask_env["AGEMATH_WEB_PORT"] = "7860"

flask_log = open("/kaggle/working/flask.log", "w")
flask_proc = subprocess.Popen(
    ["python", "-m", "agentic_math_solver", "serve"],
    env=flask_env, stdout=flask_log, stderr=subprocess.STDOUT,
)
print("Site subindo na porta 7860 (log em /kaggle/working/flask.log)")

Site subindo na porta 7860 (log em /kaggle/working/flask.log)


In [ ]:
# Célula 4 — expõe a porta 7860 publicamente via Cloudflare Tunnel (fica em primeiro plano)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:7860

2026-07-31T16:08:47Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-07-31T16:08:47Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-07-31T16:08:51Z INF +--------------------------------------------------------------------------------------------+
2026-07-31T16:08:51Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-07-31T16:08:51Z INF |  https://wright-passes-angel-slide.trycloudflare.com  